# Pairwise developmental-video coding reliability

## tl;dr

This notebook is a local-first, Colab-compatible workflow for comparing two coder CSVs. It harmonizes inconsistent schemas, validates that the files can be aligned, analyzes toy/object and looking/attention separately, applies configurable special-code exclusions, calculates agreement and Cohen's kappa, and writes time-linked disagreement logs.

The executed 5108 12-month example aligns **18,548 rows**. After one missing value is excluded in each domain, toy/object reliability is **99.9569% agreement** with **8 disagreements** and **Cohen's kappa = 0.999487**; looking/attention reliability is **89.0548% agreement** with **2,030 disagreements** and **Cohen's kappa = 0.614837**.

The workflow never uploads data by itself. Run it in the approved local Jupyter environment when compliance rules prohibit Colab or other hosted runtimes.

## Goal, scope, and quick start

The intended readers are research staff who run pairwise reliability checks and reviewers who return to the source video to adjudicate disagreements.

1. Run the setup and function-definition cells from top to bottom.
2. In **Run configuration**, set FILE_A, FILE_B, OUTPUT_DIR, and any explicit column or value mappings.
3. Run the analysis cell. Review the validation messages before interpreting the metrics.
4. Open the combined or domain-specific disagreement CSV and use video_time_hhmmss, video_time_a, and video_time_b to locate the relevant moment.

In Colab, upload or mount the two files first and assign their resulting paths. For local use, keep the files on approved storage and use ordinary filesystem paths. Filenames are not required to match, but recognizable participant and age tokens are used as a safety check when present.

### Key assumptions

- Each row represents a sampled video time or frame. Wide event-level files that only contain separate onset/offset columns must first be converted to a common sample grid; comparing event row numbers would not estimate time-based agreement.
- A file needs at least one requested coding domain and a usable video-time or frame column. Row-order alignment is disabled unless explicitly enabled.
- Input time values use one consistent unit. The included exports use milliseconds, so time_scale_to_seconds defaults to 0.001.
- Cohen's kappa is calculated only on aligned rows where both coders have non-missing, non-excluded values.
- Codes 9 and 999 are treated as special/unscorable by default. Change the exclusion sets if the study codebook assigns them another meaning.
- Numeric toy code 7 means FISH for 9-month files and POP for other known ages. If age is unknown, 7 stays unresolved and a warning is emitted.
- Study- or coder-specific labels such as FGN, FNG, or BEAD are not guessed. Supply explicit value mappings approved by the coding team.

## Setup

The implementation requires Python 3.10+, pandas, NumPy, and IPython/Jupyter. It intentionally does not require scikit-learn: the kappa calculation is implemented below from the reported marginal counts.

Local installation example:

~~~bash
python -m pip install "pandas>=2.2,<4" "numpy>=1.26,<3" jupyter
~~~

In [1]:
from __future__ import annotations

import json
import math
import re
import tempfile
from dataclasses import asdict, dataclass, field
from pathlib import Path
from typing import Any, Mapping, Sequence

import numpy as np
import pandas as pd
from IPython.display import display

pd.set_option("display.max_columns", 30)
pd.set_option("display.max_colwidth", 80)

## Input contract and canonical schema

Minimum input:

- one alignment field shared by both files: video_time is preferred; frame is the safe fallback;
- at least one shared requested domain: toy_code and/or looking_code.

Helpful optional data include both time and frame, coder/session tokens in filenames, and stable source row numbers. Extra or reordered columns are ignored. Header matching is case- and punctuation-insensitive, but it only uses explicit aliases so an onset, offset, or ordinal column cannot be mistaken for a code column.

The mapping dictionaries use canonical-name → actual-column-name. Explicit mappings always override detection. A malformed file, ambiguous alias, identity mismatch, duplicate alignment key, non-overlapping timeline, or absence of a safe alignment key produces an actionable error rather than a reliability number.

In [2]:
DEFAULT_COLUMN_ALIASES: dict[str, tuple[str, ...]] = {
    "video_time": (
        "time", "video time", "video_time", "timestamp", "video timestamp",
        "media time", "movie time", "elapsed time",
    ),
    "frame": (
        "nFrame", "frame", "frame number", "frame_number", "frame index",
    ),
    "toy_code": (
        "TOY.code01", "toy code01", "toy code", "toy_code",
        "OBJECT.code01", "object code01", "object code", "object_code",
        "Paradigm.code01", "paradigm code01", "paradigm code",
    ),
    "looking_code": (
        "LOOKING.code01", "looking code01", "looking code", "looking_code",
        "ATTENTION.code01", "attention code01", "attention code",
        "attention_code", "looking", "attention",
    ),
}

DEFAULT_VALUE_MAPS: dict[str, dict[str, str]] = {
    "toy": {
        "1": "RING",
        "2": "SQUIGGLE",
        "3": "NOISEMAKER",
        "4": "BLOCK",
        "5": "CAR",
        "6": "DRUM",
        "RING STACKER": "RING",
        "SQUIGGLE2": "SQUIGGLE",
        "NO TOY": "NO_TOY",
        "NONE": "NO_TOY",
        "NO OBJECT": "NO_TOY",
    },
    "looking": {
        "LOOK": "1",
        "LOOKING": "1",
        "YES": "1",
        "TRUE": "1",
        "NOT LOOKING": "0",
        "NO": "0",
        "FALSE": "0",
    },
}


@dataclass
class ReliabilityConfig:
    requested_domains: tuple[str, ...] = ("toy", "looking")
    excluded_codes: dict[str, set[str]] = field(
        default_factory=lambda: {
            "toy": {"9", "999"},
            "looking": {"9", "999"},
        }
    )
    age_months: int | None = None
    pair_id: str | None = None
    time_tolerance: float | None = None
    time_scale_to_seconds: float | None = 0.001
    strict_domains: bool = False
    strict_identity: bool = True
    allow_row_order_alignment: bool = False
    minimum_alignment_coverage: float = 0.90
    output_prefix: str = "pairwise_reliability"


@dataclass
class LoadedCodingFile:
    path: Path
    data: pd.DataFrame
    resolved_columns: dict[str, str | None]
    source_columns: list[str]
    profile: dict[str, Any]
    warnings: list[str]


@dataclass
class ReliabilityResult:
    summary: pd.DataFrame
    disagreements: pd.DataFrame
    aligned: pd.DataFrame
    metadata: dict[str, Any]
    output_paths: dict[str, Path] = field(default_factory=dict)


def _normalize_header(value: Any) -> str:
    return re.sub(r"[^a-z0-9]+", "", str(value).strip().casefold())


def _clean_code_token(value: Any) -> str | None:
    if value is None or pd.isna(value):
        return None
    text = str(value).strip().upper()
    if not text:
        return None
    if re.fullmatch(r"[+-]?\d+\.0+", text):
        text = text.split(".", 1)[0]
    return re.sub(r"[\s_-]+", " ", text).strip()


def _canonical_label(value: Any) -> str | None:
    token = _clean_code_token(value)
    return None if token is None else token.replace(" ", "_")


def _find_actual_column(columns: Sequence[str], requested: str) -> str | None:
    if requested in columns:
        return requested
    matches = [column for column in columns if column.casefold() == requested.casefold()]
    return matches[0] if len(matches) == 1 else None


def resolve_schema(
    columns: Sequence[str],
    column_mapping: Mapping[str, str] | None = None,
    aliases: Mapping[str, Sequence[str]] = DEFAULT_COLUMN_ALIASES,
) -> dict[str, str | None]:
    """Resolve source headers to the canonical video_time/frame/domain fields."""
    column_mapping = dict(column_mapping or {})
    unknown = set(column_mapping) - set(aliases)
    if unknown:
        raise ValueError(f"Unknown canonical mapping key(s): {sorted(unknown)}")

    resolved: dict[str, str | None] = {}
    for canonical, candidates in aliases.items():
        if canonical in column_mapping:
            actual = _find_actual_column(columns, column_mapping[canonical])
            if actual is None:
                raise ValueError(
                    f"Mapped column {column_mapping[canonical]!r} for {canonical!r} "
                    f"was not found. Available columns: {list(columns)}"
                )
            resolved[canonical] = actual
            continue

        alias_tokens = {_normalize_header(candidate) for candidate in candidates}
        matches = [column for column in columns if _normalize_header(column) in alias_tokens]
        if len(matches) > 1:
            raise ValueError(
                f"Ambiguous columns for {canonical!r}: {matches}. "
                "Provide an explicit column mapping."
            )
        resolved[canonical] = matches[0] if matches else None
    return resolved


def load_coding_file(
    path: str | Path,
    column_mapping: Mapping[str, str] | None = None,
) -> LoadedCodingFile:
    """Load one CSV, retain traceability fields, and coerce alignment values safely."""
    source_path = Path(path).expanduser().resolve()
    if not source_path.is_file():
        raise FileNotFoundError(f"Input file not found: {source_path}")
    try:
        raw = pd.read_csv(source_path, low_memory=False)
    except Exception as exc:
        raise ValueError(f"Could not parse {source_path.name} as CSV: {exc}") from exc
    if raw.empty:
        raise ValueError(f"{source_path.name} contains no data rows.")

    columns = [str(column) for column in raw.columns]
    resolved = resolve_schema(columns, column_mapping)
    warnings_list: list[str] = []
    harmonized = pd.DataFrame(index=raw.index)
    harmonized["source_row"] = raw.index + 2  # Header is row 1 in the CSV.
    for canonical in ("video_time", "frame", "toy_code", "looking_code"):
        actual = resolved[canonical]
        if actual is None:
            harmonized[canonical] = pd.Series(pd.NA, index=raw.index, dtype="object")
        else:
            harmonized[canonical] = raw[actual]

    malformed_counts: dict[str, int] = {}
    for key in ("video_time", "frame"):
        original = harmonized[key]
        numeric = pd.to_numeric(original, errors="coerce")
        malformed = int((original.notna() & numeric.isna()).sum())
        malformed_counts[key] = malformed
        harmonized[key] = numeric.astype(float)
        if malformed:
            warnings_list.append(
                f"{source_path.name}: {malformed} non-numeric {key} value(s) "
                "cannot be used for alignment."
            )

    profile = {
        "rows": int(len(raw)),
        "columns": columns,
        "resolved_columns": resolved.copy(),
        "malformed_alignment_values": malformed_counts,
        "usable_video_time_rows": int(harmonized["video_time"].notna().sum()),
        "usable_frame_rows": int(harmonized["frame"].notna().sum()),
        "usable_toy_rows": int(harmonized["toy_code"].notna().sum()),
        "usable_looking_rows": int(harmonized["looking_code"].notna().sum()),
    }
    return LoadedCodingFile(
        path=source_path,
        data=harmonized,
        resolved_columns=resolved,
        source_columns=columns,
        profile=profile,
        warnings=warnings_list,
    )

## Comparability validation and alignment

The validator separates filename evidence from data evidence:

- If both filenames expose participant IDs or ages and they conflict, strict mode stops the run.
- If neither filename exposes those tokens, the run may continue but records that identity could not be verified automatically.
- video_time uses a nearest-neighbor match bounded by a tolerance. The automatic tolerance is half the slower file's median sample interval.
- frame uses an exact one-to-one match.
- row order is available only as an explicit last resort because insertions or deletions can shift every later comparison.
- Coverage, right-row reuse, time deltas, and the selected alignment method are recorded in metadata.

This avoids the old failure mode where an unlimited nearest-time join could pair a row with a distant, unrelated moment.

In [3]:
def infer_filename_metadata(path: Path) -> dict[str, Any]:
    stem = path.stem
    participant_match = re.search(r"(?<!\d)(\d{3,})(?!\d)", stem)
    age_match = re.search(r"(?<!\d)0?(6|9|12)m(?![a-z])", stem.casefold())
    return {
        "participant_id": participant_match.group(1) if participant_match else None,
        "age_months": int(age_match.group(1)) if age_match else None,
    }


def _usable(series: pd.Series, minimum: int = 1) -> bool:
    return int(series.notna().sum()) >= minimum


def _median_positive_step(series: pd.Series) -> float | None:
    values = np.sort(series.dropna().astype(float).unique())
    if len(values) < 2:
        return None
    differences = np.diff(values)
    positive = differences[differences > 0]
    return float(np.median(positive)) if len(positive) else None


def validate_comparability(
    file_a: LoadedCodingFile,
    file_b: LoadedCodingFile,
    config: ReliabilityConfig,
) -> dict[str, Any]:
    """Choose a safe shared key/domain set and reject clearly incomparable pairs."""
    warnings_list = [*file_a.warnings, *file_b.warnings]
    requested = tuple(domain.casefold() for domain in config.requested_domains)
    invalid_domains = set(requested) - {"toy", "looking"}
    if invalid_domains:
        raise ValueError(f"Unsupported requested domain(s): {sorted(invalid_domains)}")

    shared_domains: list[str] = []
    for domain in requested:
        column = f"{domain}_code"
        available_a = _usable(file_a.data[column])
        available_b = _usable(file_b.data[column])
        if available_a and available_b:
            shared_domains.append(domain)
        else:
            message = (
                f"{domain}: unavailable for comparison "
                f"(file A present={available_a}, file B present={available_b})."
            )
            if config.strict_domains:
                raise ValueError(message)
            warnings_list.append(message + " The domain will be skipped.")
    if not shared_domains:
        raise ValueError("The files have no shared requested coding domain.")

    inferred_a = infer_filename_metadata(file_a.path)
    inferred_b = infer_filename_metadata(file_b.path)
    for key, label in (("participant_id", "participant"), ("age_months", "age")):
        value_a, value_b = inferred_a[key], inferred_b[key]
        if value_a is not None and value_b is not None and value_a != value_b:
            message = (
                f"Filename {label} mismatch: {file_a.path.name} implies {value_a}, "
                f"but {file_b.path.name} implies {value_b}."
            )
            if config.strict_identity:
                raise ValueError(message)
            warnings_list.append(message)
        elif value_a is None or value_b is None:
            warnings_list.append(
                f"Could not verify matching {label} from both filenames; "
                "confirm the pair manually."
            )

    if config.age_months is not None:
        context_age = int(config.age_months)
    elif inferred_a["age_months"] == inferred_b["age_months"]:
        context_age = inferred_a["age_months"]
    else:
        context_age = None

    if _usable(file_a.data["video_time"], 2) and _usable(file_b.data["video_time"], 2):
        alignment_method = "nearest_video_time"
        step_a = _median_positive_step(file_a.data["video_time"])
        step_b = _median_positive_step(file_b.data["video_time"])
        if config.time_tolerance is not None:
            tolerance = float(config.time_tolerance)
        else:
            steps = [step for step in (step_a, step_b) if step is not None]
            tolerance = float(math.ceil(max(steps) / 2.0)) if steps else 0.0
        if tolerance < 0:
            raise ValueError("time_tolerance must be non-negative.")
        max_start = max(file_a.data["video_time"].min(), file_b.data["video_time"].min())
        min_end = min(file_a.data["video_time"].max(), file_b.data["video_time"].max())
        if max_start > min_end + tolerance:
            raise ValueError("The video-time ranges do not overlap within the alignment tolerance.")
        alignment_key = "video_time"
    elif _usable(file_a.data["frame"]) and _usable(file_b.data["frame"]):
        alignment_method = "exact_frame"
        alignment_key = "frame"
        tolerance = 0.0
        step_a = step_b = None
    elif config.allow_row_order_alignment:
        alignment_method = "row_order"
        alignment_key = "row_order"
        tolerance = 0.0
        step_a = step_b = None
        warnings_list.append(
            "No shared time/frame field was usable. Row-order alignment was explicitly "
            "enabled; results are unsafe if either file has inserted or deleted rows."
        )
    else:
        raise ValueError(
            "No safe shared alignment field. Map a video-time alias, provide frame "
            "columns in both files, or explicitly enable row-order alignment."
        )

    if alignment_key in {"video_time", "frame"}:
        for label, loaded in (("A", file_a), ("B", file_b)):
            duplicate_count = int(
                loaded.data.loc[loaded.data[alignment_key].notna(), alignment_key]
                .duplicated()
                .sum()
            )
            if duplicate_count:
                raise ValueError(
                    f"File {label} has {duplicate_count} duplicate {alignment_key} "
                    "value(s); resolve them before comparison."
                )

    participant = (
        inferred_a["participant_id"]
        if inferred_a["participant_id"] == inferred_b["participant_id"]
        else None
    )
    age_for_id = context_age
    inferred_pair_id = (
        f"{participant}_{age_for_id}m"
        if participant and age_for_id
        else participant or "pair"
    )
    return {
        "shared_domains": shared_domains,
        "alignment_method": alignment_method,
        "alignment_key": alignment_key,
        "time_tolerance": tolerance,
        "median_step_a": step_a,
        "median_step_b": step_b,
        "inferred_a": inferred_a,
        "inferred_b": inferred_b,
        "age_months": context_age,
        "pair_id": config.pair_id or inferred_pair_id,
        "warnings": warnings_list,
    }


def _suffix_columns(data: pd.DataFrame, suffix: str) -> pd.DataFrame:
    columns = [
        "source_row", "video_time", "frame", "toy_code", "looking_code",
    ]
    return data[columns].rename(columns={column: f"{column}_{suffix}" for column in columns})


def align_coding_files(
    file_a: LoadedCodingFile,
    file_b: LoadedCodingFile,
    validation: dict[str, Any],
    config: ReliabilityConfig,
) -> tuple[pd.DataFrame, dict[str, Any], list[str]]:
    """Align two harmonized files and retain both sources' time/frame references."""
    method = validation["alignment_method"]
    warnings_list: list[str] = []
    a = _suffix_columns(file_a.data, "a")
    b = _suffix_columns(file_b.data, "b")

    if method == "nearest_video_time":
        key_a, key_b = "video_time_a", "video_time_b"
        tolerance = float(validation["time_tolerance"])
        a_valid = a[a[key_a].notna()].sort_values(key_a).copy()
        b_valid = b[b[key_b].notna()].sort_values(key_b).copy()
        a_low, a_high = a_valid[key_a].min(), a_valid[key_a].max()
        b_low, b_high = b_valid[key_b].min(), b_valid[key_b].max()
        a_candidates = a_valid[
            a_valid[key_a].between(b_low - tolerance, b_high + tolerance)
        ]
        b_candidates = b_valid[
            b_valid[key_b].between(a_low - tolerance, a_high + tolerance)
        ]
        aligned = pd.merge_asof(
            a_candidates,
            b_valid,
            left_on=key_a,
            right_on=key_b,
            direction="nearest",
            tolerance=tolerance,
        )
        aligned = aligned[aligned["source_row_b"].notna()].copy()
        aligned["alignment_value"] = aligned[key_a]
        aligned["alignment_delta"] = (aligned[key_a] - aligned[key_b]).abs()
    elif method == "exact_frame":
        key_a, key_b = "frame_a", "frame_b"
        a_candidates = a[a[key_a].notna()].copy()
        b_candidates = b[b[key_b].notna()].copy()
        aligned = a_candidates.merge(
            b_candidates,
            left_on=key_a,
            right_on=key_b,
            how="inner",
            validate="one_to_one",
        )
        aligned["alignment_value"] = aligned[key_a]
        aligned["alignment_delta"] = 0.0
    else:
        a_candidates = a.reset_index(drop=True).copy()
        b_candidates = b.reset_index(drop=True).copy()
        a_candidates["_row_order"] = np.arange(len(a_candidates))
        b_candidates["_row_order"] = np.arange(len(b_candidates))
        aligned = a_candidates.merge(
            b_candidates, on="_row_order", how="inner", validate="one_to_one"
        )
        aligned["alignment_value"] = aligned["_row_order"]
        aligned["alignment_delta"] = 0.0

    if aligned.empty:
        raise ValueError("No rows aligned. Check the pair, column mapping, and tolerance.")

    aligned["video_time"] = aligned["video_time_a"].combine_first(aligned["video_time_b"])
    aligned["time_delta"] = aligned["video_time_a"] - aligned["video_time_b"]
    reused_b = int(aligned["source_row_b"].duplicated().sum())
    overlap_coverage_a = (
        len(aligned) / len(a_candidates) if len(a_candidates) else 0.0
    )
    unique_b = int(aligned["source_row_b"].nunique())
    overlap_coverage_b = (
        unique_b / len(b_candidates) if len(b_candidates) else 0.0
    )
    if method == "nearest_video_time":
        total_rows_a, total_rows_b = len(a_valid), len(b_valid)
    else:
        total_rows_a, total_rows_b = len(a_candidates), len(b_candidates)
    full_file_coverage_a = len(aligned) / total_rows_a if total_rows_a else 0.0
    full_file_coverage_b = unique_b / total_rows_b if total_rows_b else 0.0
    if full_file_coverage_a < config.minimum_alignment_coverage:
        warnings_list.append(
            f"Only {full_file_coverage_a:.1%} of usable file-A key rows aligned; "
            "inspect time ranges "
            "and tolerance before using the result."
        )
    if full_file_coverage_b < config.minimum_alignment_coverage:
        warnings_list.append(
            f"Only {full_file_coverage_b:.1%} of usable file-B key rows were "
            "represented; inspect time ranges and tolerance before using the result."
        )
    if reused_b:
        warnings_list.append(
            f"{reused_b} aligned row(s) reuse a file-B row. This can occur when sampling "
            "rates differ and should be reviewed."
        )

    metadata = {
        "method": method,
        "key": validation["alignment_key"],
        "time_tolerance": validation["time_tolerance"],
        "median_step_a": validation["median_step_a"],
        "median_step_b": validation["median_step_b"],
        "candidate_rows_a": int(len(a_candidates)),
        "candidate_rows_b": int(len(b_candidates)),
        "aligned_rows": int(len(aligned)),
        "overlap_coverage_a": float(overlap_coverage_a),
        "overlap_coverage_b_unique": float(overlap_coverage_b),
        "full_file_coverage_a": float(full_file_coverage_a),
        "full_file_coverage_b_unique": float(full_file_coverage_b),
        "reused_file_b_rows": reused_b,
        "maximum_alignment_delta": float(aligned["alignment_delta"].max()),
    }
    return aligned.reset_index(drop=True), metadata, warnings_list

## Reliability logic

For a domain, let \(N\) be the number of aligned rows remaining after missing and special-code exclusions. Let \(A\) be the count where the two normalized codes agree and \(D\) the count where they differ.

\[
\text{Agreement percentage} = P_o \times 100
= \frac{A}{N} \times 100
\]

\[
\text{Disagreement percentage}
= \frac{D}{N} \times 100
= (1-P_o)\times 100
\]

For each code category \(c\), let \(p_A(c)\) and \(p_B(c)\) be the observed proportions assigned by coders A and B. Chance-expected agreement and Cohen's kappa are:

\[
P_e = \sum_{c \in C} p_A(c)\,p_B(c)
\]

\[
\kappa = \frac{P_o-P_e}{1-P_e}
\]

Worked example: among 100 eligible observations, suppose 45 are 1/1, 40 are 0/0, 5 are 1/0, and 10 are 0/1. Then \(A=85\), \(D=15\), \(P_o=0.85\), \(P_e=(0.50\times0.55)+(0.50\times0.45)=0.50\), and \(\kappa=(0.85-0.50)/(1-0.50)=0.70\). The code cell below calculates the same values, so the displayed formula and implementation are directly reconciled.

Kappa is undefined when no eligible observations remain or when \(P_e=1\) (both marginals contain no variation). The summary reports a status instead of fabricating a number.

In [4]:
def normalize_code(
    value: Any,
    domain: str,
    age_months: int | None,
    custom_mapping: Mapping[Any, Any] | None = None,
) -> str | None:
    """Normalize one value with explicit mappings before approved defaults."""
    token = _clean_code_token(value)
    if token is None:
        return None
    custom = {
        _clean_code_token(key): _canonical_label(target)
        for key, target in (custom_mapping or {}).items()
    }
    if token in custom:
        return custom[token]

    defaults = {
        _clean_code_token(key): _canonical_label(target)
        for key, target in DEFAULT_VALUE_MAPS.get(domain, {}).items()
    }
    if domain == "toy" and token == "7":
        if age_months == 9:
            return "FISH"
        if age_months is not None:
            return "POP"
        return "7"
    return defaults.get(token, _canonical_label(token))


def calculate_cohens_kappa(
    coder_a: pd.Series,
    coder_b: pd.Series,
) -> dict[str, Any]:
    """Calculate unweighted Cohen's kappa from paired, already-eligible values."""
    if len(coder_a) != len(coder_b):
        raise ValueError("Kappa inputs must have equal lengths.")
    n = int(len(coder_a))
    if n == 0:
        return {
            "observed_agreement": np.nan,
            "expected_agreement": np.nan,
            "cohen_kappa": np.nan,
            "kappa_status": "undefined_no_eligible_rows",
        }
    observed = float((coder_a.to_numpy() == coder_b.to_numpy()).mean())
    categories = sorted(set(coder_a.astype(str)) | set(coder_b.astype(str)))
    expected = float(
        sum(
            (coder_a.astype(str) == category).mean()
            * (coder_b.astype(str) == category).mean()
            for category in categories
        )
    )
    if math.isclose(1.0 - expected, 0.0, abs_tol=1e-12):
        kappa = np.nan
        status = "undefined_expected_agreement_is_one"
    else:
        kappa = float((observed - expected) / (1.0 - expected))
        status = "ok"
    return {
        "observed_agreement": observed,
        "expected_agreement": expected,
        "cohen_kappa": kappa,
        "kappa_status": status,
    }


worked_a = pd.Series(["1"] * 45 + ["1"] * 5 + ["0"] * 10 + ["0"] * 40)
worked_b = pd.Series(["1"] * 45 + ["0"] * 5 + ["1"] * 10 + ["0"] * 40)
worked_metrics = calculate_cohens_kappa(worked_a, worked_b)
display(
    pd.DataFrame(
        [{
            "eligible_rows": len(worked_a),
            "agreement_count": int((worked_a == worked_b).sum()),
            "disagreement_count": int((worked_a != worked_b).sum()),
            "agreement_pct": 100 * worked_metrics["observed_agreement"],
            "disagreement_pct": 100 * (1 - worked_metrics["observed_agreement"]),
            "expected_agreement": worked_metrics["expected_agreement"],
            "cohen_kappa": worked_metrics["cohen_kappa"],
        }]
    )
)

,eligible_rows,agreement_count,disagreement_count,agreement_pct,disagreement_pct,expected_agreement,cohen_kappa
0,100,85,15,85.0,15.0,0.5,0.7


## Domain comparison and time-linked disagreements

Toy/object and looking/attention are normalized and scored independently. A missing or excluded value removes that row only from its own domain, so a special looking code does not suppress a valid toy comparison.

Best case: both files contain video time. Each disagreement keeps a review time plus both original times, the signed time delta, both frames, both CSV row numbers, raw codes, and normalized codes. With the default millisecond scale it also includes seconds and HH:MM:SS.mmm.

Fallback: if time is unavailable but both files have frames, exact-frame alignment is used and the time fields remain blank. If only one file has time but both have frame, frame alignment is used and the available time is retained. Inconsistent time headers are resolved through aliases or explicit mappings. Without time or frame, the run stops unless the user explicitly accepts row-order alignment.

In [5]:
DISAGREEMENT_COLUMNS = [
    "pair_id", "domain", "file_a", "file_b", "alignment_method",
    "alignment_value", "alignment_delta", "video_time", "video_time_a",
    "video_time_b", "time_delta", "video_time_seconds", "video_time_hhmmss",
    "time_reference_status", "frame_a", "frame_b", "source_row_a",
    "source_row_b", "raw_code_a", "raw_code_b", "normalized_code_a",
    "normalized_code_b",
]


def _format_hhmmss(seconds: Any) -> str | None:
    if seconds is None or pd.isna(seconds):
        return None
    total = float(seconds)
    sign = "-" if total < 0 else ""
    total = abs(total)
    hours = int(total // 3600)
    minutes = int((total % 3600) // 60)
    remainder = total % 60
    return f"{sign}{hours:02d}:{minutes:02d}:{remainder:06.3f}"


def compare_domain(
    aligned: pd.DataFrame,
    domain: str,
    file_a: LoadedCodingFile,
    file_b: LoadedCodingFile,
    validation: dict[str, Any],
    alignment_metadata: dict[str, Any],
    config: ReliabilityConfig,
    value_mapping_a: Mapping[Any, Any] | None = None,
    value_mapping_b: Mapping[Any, Any] | None = None,
) -> tuple[dict[str, Any], pd.DataFrame, pd.DataFrame, list[str]]:
    """Compare one domain and return its summary, log, enriched rows, and warnings."""
    warnings_list: list[str] = []
    raw_a_name, raw_b_name = f"{domain}_code_a", f"{domain}_code_b"
    work = aligned.copy()
    work["normalized_code_a"] = work[raw_a_name].map(
        lambda value: normalize_code(
            value, domain, validation["age_months"], value_mapping_a
        )
    )
    work["normalized_code_b"] = work[raw_b_name].map(
        lambda value: normalize_code(
            value, domain, validation["age_months"], value_mapping_b
        )
    )

    if domain == "toy":
        raw_tokens = {
            token
            for token in (
                work[raw_a_name].map(_clean_code_token).tolist()
                + work[raw_b_name].map(_clean_code_token).tolist()
            )
            if token is not None
        }
        if "7" in raw_tokens and validation["age_months"] is None:
            warnings_list.append(
                "Toy code 7 is present but age is unknown; provide age_months so it "
                "can be normalized to FISH (9m) or POP (other known ages)."
            )
        normalized_values = set(work["normalized_code_a"].dropna()) | set(
            work["normalized_code_b"].dropna()
        )
        if validation["age_months"] == 9 and "POP" in normalized_values:
            warnings_list.append("POP appears in a 9-month pair; confirm the codebook exception.")
        if (
            validation["age_months"] is not None
            and validation["age_months"] != 9
            and "FISH" in normalized_values
        ):
            warnings_list.append(
                "FISH appears outside a 9-month pair; confirm the codebook exception."
            )

    excluded = {
        _canonical_label(value)
        for value in config.excluded_codes.get(domain, set())
        if _canonical_label(value) is not None
    }
    missing_mask = work["normalized_code_a"].isna() | work["normalized_code_b"].isna()
    special_mask = (
        work["normalized_code_a"].isin(excluded)
        | work["normalized_code_b"].isin(excluded)
    )
    eligible_mask = ~(missing_mask | special_mask)
    eligible = work.loc[eligible_mask].copy()
    eligible["is_agreement"] = (
        eligible["normalized_code_a"] == eligible["normalized_code_b"]
    )

    metrics = calculate_cohens_kappa(
        eligible["normalized_code_a"], eligible["normalized_code_b"]
    )
    agreement_count = int(eligible["is_agreement"].sum())
    disagreement_count = int((~eligible["is_agreement"]).sum())
    n = int(len(eligible))
    summary = {
        "pair_id": validation["pair_id"],
        "domain": domain,
        "file_a": file_a.path.name,
        "file_b": file_b.path.name,
        "alignment_method": validation["alignment_method"],
        "time_tolerance": validation["time_tolerance"],
        "aligned_rows": int(len(work)),
        "eligible_rows": n,
        "excluded_missing_count": int(missing_mask.sum()),
        "excluded_special_count": int((~missing_mask & special_mask).sum()),
        "excluded_total_count": int((missing_mask | special_mask).sum()),
        "agreement_count": agreement_count,
        "disagreement_count": disagreement_count,
        "agreement_pct": round(100 * metrics["observed_agreement"], 6)
        if n else np.nan,
        "disagreement_pct": round(100 * (1 - metrics["observed_agreement"]), 6)
        if n else np.nan,
        "expected_agreement": round(metrics["expected_agreement"], 9)
        if n else np.nan,
        "cohen_kappa": round(metrics["cohen_kappa"], 9)
        if metrics["kappa_status"] == "ok" else np.nan,
        "kappa_status": metrics["kappa_status"],
    }

    disagreement_rows = eligible.loc[~eligible["is_agreement"]].copy()
    if config.time_scale_to_seconds is not None:
        seconds = (
            disagreement_rows["video_time"] * config.time_scale_to_seconds
        ).round(6)
    else:
        seconds = pd.Series(pd.NA, index=disagreement_rows.index, dtype="object")
    both_time = (
        disagreement_rows["video_time_a"].notna()
        & disagreement_rows["video_time_b"].notna()
    )
    a_only = (
        disagreement_rows["video_time_a"].notna()
        & disagreement_rows["video_time_b"].isna()
    )
    b_only = (
        disagreement_rows["video_time_a"].isna()
        & disagreement_rows["video_time_b"].notna()
    )
    log = pd.DataFrame(index=disagreement_rows.index)
    log["pair_id"] = validation["pair_id"]
    log["domain"] = domain
    log["file_a"] = file_a.path.name
    log["file_b"] = file_b.path.name
    log["alignment_method"] = validation["alignment_method"]
    for column in (
        "alignment_value", "alignment_delta", "video_time", "video_time_a",
        "video_time_b", "time_delta", "frame_a", "frame_b", "source_row_a",
        "source_row_b",
    ):
        log[column] = disagreement_rows[column]
    log["video_time_seconds"] = seconds
    log["video_time_hhmmss"] = seconds.map(_format_hhmmss)
    log["time_reference_status"] = np.select(
        [both_time, a_only, b_only],
        ["both", "file_a_only", "file_b_only"],
        default="missing",
    )
    log["raw_code_a"] = disagreement_rows[raw_a_name]
    log["raw_code_b"] = disagreement_rows[raw_b_name]
    log["normalized_code_a"] = disagreement_rows["normalized_code_a"]
    log["normalized_code_b"] = disagreement_rows["normalized_code_b"]
    log = log.reindex(columns=DISAGREEMENT_COLUMNS).reset_index(drop=True)

    enriched = work[["source_row_a", "source_row_b"]].copy()
    enriched[f"{domain}_normalized_a"] = work["normalized_code_a"]
    enriched[f"{domain}_normalized_b"] = work["normalized_code_b"]
    enriched[f"{domain}_eligible"] = eligible_mask
    return summary, log, enriched, warnings_list

## Report writer and end-to-end runner

The runner owns the full audit trail: loading, schema resolution, comparability checks, alignment, domain comparisons, warnings, and report writing. Summary percentages are numeric values on a 0–100 scale, which keeps the CSV machine-readable.

In [6]:
def _jsonable(value: Any) -> Any:
    if isinstance(value, Path):
        return str(value)
    if isinstance(value, set):
        return sorted(_jsonable(item) for item in value)
    if isinstance(value, tuple):
        return [_jsonable(item) for item in value]
    if isinstance(value, list):
        return [_jsonable(item) for item in value]
    if isinstance(value, dict):
        return {str(key): _jsonable(item) for key, item in value.items()}
    if isinstance(value, np.generic):
        return _jsonable(value.item())
    if isinstance(value, float) and not math.isfinite(value):
        return None
    return value


def write_reliability_reports(
    result: ReliabilityResult,
    output_dir: str | Path,
    prefix: str,
) -> dict[str, Path]:
    output_path = Path(output_dir).expanduser().resolve()
    output_path.mkdir(parents=True, exist_ok=True)
    safe_prefix = re.sub(r"[^A-Za-z0-9._-]+", "_", prefix).strip("_")
    if not safe_prefix:
        raise ValueError("output_prefix must contain at least one filename-safe character.")

    paths: dict[str, Path] = {}
    paths["summary"] = output_path / f"{safe_prefix}_summary.csv"
    result.summary.to_csv(paths["summary"], index=False)
    paths["disagreement_log"] = output_path / f"{safe_prefix}_disagreement_log.csv"
    result.disagreements.to_csv(paths["disagreement_log"], index=False)

    for domain in result.summary["domain"]:
        domain_summary = result.summary[result.summary["domain"] == domain]
        domain_log = result.disagreements[result.disagreements["domain"] == domain]
        paths[f"{domain}_summary"] = output_path / f"{safe_prefix}_{domain}_summary.csv"
        paths[f"{domain}_disagreement_log"] = (
            output_path / f"{safe_prefix}_{domain}_disagreement_log.csv"
        )
        domain_summary.to_csv(paths[f"{domain}_summary"], index=False)
        domain_log.to_csv(paths[f"{domain}_disagreement_log"], index=False)

    paths["metadata"] = output_path / f"{safe_prefix}_metadata.json"
    result.output_paths = paths
    result.metadata["output_paths"] = {key: str(path) for key, path in paths.items()}
    with paths["metadata"].open("w", encoding="utf-8") as stream:
        json.dump(_jsonable(result.metadata), stream, indent=2, allow_nan=False)
    return paths


def run_pairwise_reliability(
    file_a_path: str | Path,
    file_b_path: str | Path,
    *,
    config: ReliabilityConfig | None = None,
    column_mapping_a: Mapping[str, str] | None = None,
    column_mapping_b: Mapping[str, str] | None = None,
    value_mapping_a: Mapping[str, Mapping[Any, Any]] | None = None,
    value_mapping_b: Mapping[str, Mapping[Any, Any]] | None = None,
    output_dir: str | Path | None = None,
    write_outputs: bool = True,
) -> ReliabilityResult:
    """Run a validated pairwise analysis for every shared requested domain."""
    config = config or ReliabilityConfig()
    value_mapping_a = dict(value_mapping_a or {})
    value_mapping_b = dict(value_mapping_b or {})
    print(f"Loading file A: {Path(file_a_path).name}")
    print(f"Loading file B: {Path(file_b_path).name}")
    loaded_a = load_coding_file(file_a_path, column_mapping_a)
    loaded_b = load_coding_file(file_b_path, column_mapping_b)
    validation = validate_comparability(loaded_a, loaded_b, config)
    aligned, alignment_metadata, alignment_warnings = align_coding_files(
        loaded_a, loaded_b, validation, config
    )

    summaries: list[dict[str, Any]] = []
    logs: list[pd.DataFrame] = []
    enriched = aligned.copy()
    domain_warnings: list[str] = []
    for domain in validation["shared_domains"]:
        summary, log, domain_rows, current_warnings = compare_domain(
            aligned=aligned,
            domain=domain,
            file_a=loaded_a,
            file_b=loaded_b,
            validation=validation,
            alignment_metadata=alignment_metadata,
            config=config,
            value_mapping_a=value_mapping_a.get(domain),
            value_mapping_b=value_mapping_b.get(domain),
        )
        summaries.append(summary)
        logs.append(log)
        enriched = enriched.merge(
            domain_rows, on=["source_row_a", "source_row_b"], how="left", validate="one_to_one"
        )
        domain_warnings.extend(current_warnings)

    summary_frame = pd.DataFrame(summaries)
    disagreement_frame = (
        pd.concat(logs, ignore_index=True).reindex(columns=DISAGREEMENT_COLUMNS)
        if logs
        else pd.DataFrame(columns=DISAGREEMENT_COLUMNS)
    )
    all_warnings = list(
        dict.fromkeys(validation["warnings"] + alignment_warnings + domain_warnings)
    )
    metadata = {
        "workflow_version": "2.0",
        "file_a": {
            "path": str(loaded_a.path),
            "profile": loaded_a.profile,
        },
        "file_b": {
            "path": str(loaded_b.path),
            "profile": loaded_b.profile,
        },
        "configuration": asdict(config),
        "column_mapping_a": dict(column_mapping_a or {}),
        "column_mapping_b": dict(column_mapping_b or {}),
        "value_mapping_a": value_mapping_a,
        "value_mapping_b": value_mapping_b,
        "validation": {
            key: value
            for key, value in validation.items()
            if key != "warnings"
        },
        "alignment": alignment_metadata,
        "warnings": all_warnings,
        "domain_summaries": summaries,
    }
    result = ReliabilityResult(
        summary=summary_frame,
        disagreements=disagreement_frame,
        aligned=enriched,
        metadata=metadata,
    )

    print(
        f"Aligned {len(aligned):,} rows using {validation['alignment_method']} "
        f"(tolerance={validation['time_tolerance']})."
    )
    for row in summaries:
        print(
            f"{row['domain']}: {row['agreement_pct']:.4f}% agreement, "
            f"{row['disagreement_count']:,} disagreements, "
            f"kappa={row['cohen_kappa'] if pd.notna(row['cohen_kappa']) else 'undefined'}."
        )
    for message in all_warnings:
        print(f"WARNING: {message}")

    if write_outputs:
        destination = output_dir or Path.cwd() / "reliability_outputs"
        paths = write_reliability_reports(result, destination, config.output_prefix)
        print(f"Wrote {len(paths)} report file(s) to {Path(destination).resolve()}")
    else:
        print("Report writing is disabled for this run.")
    return result

## Run configuration

The historical example uses the 5108 12-month pair because both requested domains are present and its labels follow the current codebook. Change only this section for a routine run.

Column mappings are needed only when aliases do not resolve a header. Value mappings must be reviewed against the study codebook; they are applied separately to coder A and coder B. For example:

~~~python
COLUMN_MAPPING_A = {"video_time": "MovieTime", "toy_code": "Object.code01"}
VALUE_MAPPING_B = {"toy": {"coder-specific label": "approved canonical label"}}
~~~

In [7]:
repository_root = Path.cwd()
if (repository_root / "projects" / "Reliability Coding").is_dir():
    PROJECT_DIR = repository_root / "projects" / "Reliability Coding"
else:
    PROJECT_DIR = repository_root

FILE_A = PROJECT_DIR / "5108_12m_OIX_Felix.csv"
FILE_B = PROJECT_DIR / "5108_12m_OIX_Tayah.csv"
OUTPUT_DIR = PROJECT_DIR / "reliability_outputs"

COLUMN_MAPPING_A: dict[str, str] = {}
COLUMN_MAPPING_B: dict[str, str] = {}
VALUE_MAPPING_A: dict[str, dict[Any, Any]] = {}
VALUE_MAPPING_B: dict[str, dict[Any, Any]] = {}

CONFIG = ReliabilityConfig(
    requested_domains=("toy", "looking"),
    excluded_codes={"toy": {"9", "999"}, "looking": {"9", "999"}},
    age_months=None,  # Inferred as 12 from both example filenames.
    time_tolerance=None,  # Auto: half the slower median sampling interval.
    time_scale_to_seconds=0.001,  # Source export uses milliseconds.
    strict_domains=True,
    strict_identity=True,
    allow_row_order_alignment=False,
    minimum_alignment_coverage=0.90,
    output_prefix="5108_12m_pairwise_reliability",
)

In [8]:
if FILE_A.is_file() and FILE_B.is_file():
    result = run_pairwise_reliability(
        FILE_A,
        FILE_B,
        config=CONFIG,
        column_mapping_a=COLUMN_MAPPING_A,
        column_mapping_b=COLUMN_MAPPING_B,
        value_mapping_a=VALUE_MAPPING_A,
        value_mapping_b=VALUE_MAPPING_B,
        output_dir=OUTPUT_DIR,
        write_outputs=True,
    )
    display(result.summary)
    display(result.disagreements.head(10))
    display(
        pd.DataFrame(
            [{"report": key, "path": str(path)} for key, path in result.output_paths.items()]
        )
    )
else:
    result = None
    print("Set FILE_A and FILE_B to two available CSV paths, then rerun this cell.")

Loading file A: 5108_12m_OIX_Felix.csv
Loading file B: 5108_12m_OIX_Tayah.csv


Aligned 18,548 rows using nearest_video_time (tolerance=17.0).
toy: 99.9569% agreement, 8 disagreements, kappa=0.999486925.
looking: 89.0548% agreement, 2,030 disagreements, kappa=0.614837162.
Wrote 7 report file(s) to /Users/namomac/esd-redcap-metadata-watcher/projects/Reliability Coding/reliability_outputs


,pair_id,domain,file_a,file_b,alignment_method,time_tolerance,aligned_rows,eligible_rows,excluded_missing_count,excluded_special_count,excluded_total_count,agreement_count,disagreement_count,agreement_pct,disagreement_pct,expected_agreement,cohen_kappa,kappa_status
0,5108_12m,toy,5108_12m_OIX_Felix.csv,5108_12m_OIX_Tayah.csv,nearest_video_time,17.0,18548,18547,1,0,1,18539,8,99.956866,0.043134,0.159312,0.999487,ok
1,5108_12m,looking,5108_12m_OIX_Felix.csv,5108_12m_OIX_Tayah.csv,nearest_video_time,17.0,18548,18547,1,0,1,16517,2030,89.054834,10.945166,0.715830,0.614837,ok


,pair_id,domain,file_a,file_b,alignment_method,alignment_value,alignment_delta,video_time,video_time_a,video_time_b,time_delta,video_time_seconds,video_time_hhmmss,time_reference_status,frame_a,frame_b,source_row_a,source_row_b,raw_code_a,raw_code_b,normalized_code_a,normalized_code_b
0,5108_12m,toy,5108_12m_OIX_Felix.csv,5108_12m_OIX_Tayah.csv,nearest_video_time,142923.0,1.0,142923.0,142923.0,142924.0,-1.0,142.923,00:02:22.923,both,3992.0,3991.0,3993,3992,RING,SQUIGGLE,RING,SQUIGGLE
1,5108_12m,toy,5108_12m_OIX_Felix.csv,5108_12m_OIX_Tayah.csv,nearest_video_time,212355.0,1.0,212355.0,212355.0,212356.0,-1.0,212.355,00:03:32.355,both,6096.0,6095.0,6097,6096,SQUIGGLE,NOISEMAKER,SQUIGGLE,NOISEMAKER
2,5108_12m,toy,5108_12m_OIX_Felix.csv,5108_12m_OIX_Tayah.csv,nearest_video_time,212388.0,1.0,212388.0,212388.0,212389.0,-1.0,212.388,00:03:32.388,both,6097.0,6096.0,6098,6097,SQUIGGLE,NOISEMAKER,SQUIGGLE,NOISEMAKER
3,5108_12m,toy,5108_12m_OIX_Felix.csv,5108_12m_OIX_Tayah.csv,nearest_video_time,347952.0,1.0,347952.0,347952.0,347953.0,-1.0,347.952,00:05:47.952,both,10205.0,10204.0,10206,10205,BLOCK,NOISEMAKER,BLOCK,NOISEMAKER
4,5108_12m,toy,5108_12m_OIX_Felix.csv,5108_12m_OIX_Tayah.csv,nearest_video_time,410751.0,1.0,410751.0,410751.0,410752.0,-1.0,410.751,00:06:50.751,both,12108.0,12107.0,12109,12108,CAR,BLOCK,CAR,BLOCK
5,5108_12m,toy,5108_12m_OIX_Felix.csv,5108_12m_OIX_Tayah.csv,nearest_video_time,410784.0,1.0,410784.0,410784.0,410785.0,-1.0,410.784,00:06:50.784,both,12109.0,12108.0,12110,12109,CAR,BLOCK,CAR,BLOCK
6,5108_12m,toy,5108_12m_OIX_Felix.csv,5108_12m_OIX_Tayah.csv,nearest_video_time,560868.0,1.0,560868.0,560868.0,560869.0,-1.0,560.868,00:09:20.868,both,16657.0,16656.0,16658,16657,DRUM,POP,DRUM,POP
7,5108_12m,toy,5108_12m_OIX_Felix.csv,5108_12m_OIX_Tayah.csv,nearest_video_time,560901.0,1.0,560901.0,560901.0,560902.0,-1.0,560.901,00:09:20.901,both,16658.0,16657.0,16659,16658,DRUM,POP,DRUM,POP
8,5108_12m,looking,5108_12m_OIX_Felix.csv,5108_12m_OIX_Tayah.csv,nearest_video_time,15444.0,1.0,15444.0,15444.0,15445.0,-1.0,15.444,00:00:15.444,both,129.0,128.0,130,129,0.0,1.0,0,1
9,5108_12m,looking,5108_12m_OIX_Felix.csv,5108_12m_OIX_Tayah.csv,nearest_video_time,22803.0,1.0,22803.0,22803.0,22804.0,-1.0,22.803,00:00:22.803,both,352.0,351.0,353,352,0.0,1.0,0,1


,report,path
0,summary,/Users/namomac/esd-redcap-metadata-watcher/projects/Reliability Coding/relia...
1,disagreement_log,/Users/namomac/esd-redcap-metadata-watcher/projects/Reliability Coding/relia...
2,toy_summary,/Users/namomac/esd-redcap-metadata-watcher/projects/Reliability Coding/relia...
3,toy_disagreement_log,/Users/namomac/esd-redcap-metadata-watcher/projects/Reliability Coding/relia...
4,looking_summary,/Users/namomac/esd-redcap-metadata-watcher/projects/Reliability Coding/relia...
5,looking_disagreement_log,/Users/namomac/esd-redcap-metadata-watcher/projects/Reliability Coding/relia...
6,metadata,/Users/namomac/esd-redcap-metadata-watcher/projects/Reliability Coding/relia...


## Exact output specification

| Output | Purpose and key columns | Primary user | Operational value |
|---|---|---|---|
| PREFIX_summary.csv | One row per domain. Includes files, alignment method/tolerance, aligned and eligible rows, mutually exclusive missing/special exclusion counts, agreements, disagreements, numeric percentages, expected agreement, kappa, and kappa status. | Analyst, PI, data manager | The compact auditable result for tracking reliability. |
| PREFIX_toy_summary.csv | The toy/object row from the combined summary. | Toy/object coding lead | Can be routed without exposing unrelated domain results. |
| PREFIX_looking_summary.csv | The looking/attention row from the combined summary. | Looking/attention coding lead | Keeps attention reliability separate from object reliability. |
| PREFIX_disagreement_log.csv | All eligible disagreements. Key fields are domain, video_time_hhmmss, both source times/frames/rows, raw codes, normalized codes, alignment delta, and source filenames. | Adjudicator, coder trainer | Sends reviewers directly to the discrepant moment and preserves provenance. |
| PREFIX_toy_disagreement_log.csv | Toy/object disagreements only, with the same traceability fields. | Toy/object adjudicator | Domain-specific adjudication queue. |
| PREFIX_looking_disagreement_log.csv | Looking/attention disagreements only. | Attention adjudicator | Domain-specific adjudication queue. |
| PREFIX_metadata.json | Source schemas and resolved mappings, row profiles, configuration, filename inference, alignment diagnostics, warnings, summaries, and output paths. | Data manager, auditor, future maintainer | Explains exactly how the CSV results were produced and flags comparability limitations. |

The console reports selected files, aligned row count, method and tolerance, domain metrics, every warning, and the destination directory. Empty disagreement logs are still written with their expected headers.

## Automated checks

These compact tests cover the highest-risk behavior: inconsistent/reordered headers, extra columns, bounded time alignment, numeric/text toy normalization, the 9-month FISH rule, special exclusions, separate domain metrics, timestamp retention, frame fallback, report writing, and a malformed file with no safe alignment key.

In [9]:
with tempfile.TemporaryDirectory() as temporary_directory:
    temp = Path(temporary_directory)
    synthetic_a = pd.DataFrame(
        {
            "extra": ["x"] * 5,
            "attention": [1, 0, 1, 1, 1],
            "object_code": [1, 2, 9, 7, 1],
            "timestamp": [0, 40, 80, 120, 160],
            "frame_number": [1, 2, 3, 4, 5],
        }
    )
    synthetic_b = pd.DataFrame(
        {
            "nFrame": [1, 2, 3, 4, 5],
            "video time": [1, 41, 81, 121, 161],
            "Paradigm.code01": ["RING", "SQUIGGLE", "RING", "FISH", "2"],
            "Looking.code01": [1, 0, 0, 999, 1],
            "unused": [None] * 5,
        }
    )
    path_a = temp / "9001_9m_CoderA.csv"
    path_b = temp / "9001_9m_CoderB.csv"
    synthetic_a.to_csv(path_a, index=False)
    synthetic_b.to_csv(path_b, index=False)
    test_config = ReliabilityConfig(
        strict_domains=True,
        output_prefix="synthetic_pair",
    )
    synthetic_result = run_pairwise_reliability(
        path_a,
        path_b,
        config=test_config,
        output_dir=temp / "reports",
        write_outputs=True,
    )
    by_domain = synthetic_result.summary.set_index("domain")
    assert by_domain.loc["toy", "eligible_rows"] == 4
    assert by_domain.loc["toy", "disagreement_count"] == 1
    assert by_domain.loc["toy", "agreement_pct"] == 75.0
    assert by_domain.loc["looking", "eligible_rows"] == 4
    assert by_domain.loc["looking", "disagreement_count"] == 1
    assert by_domain.loc["looking", "agreement_pct"] == 75.0
    assert synthetic_result.disagreements["video_time_hhmmss"].notna().all()
    assert set(synthetic_result.output_paths) == {
        "summary", "disagreement_log", "toy_summary", "toy_disagreement_log",
        "looking_summary", "looking_disagreement_log", "metadata",
    }
    assert all(path.is_file() for path in synthetic_result.output_paths.values())

    frame_a = temp / "frame_coder_a.csv"
    frame_b = temp / "frame_coder_b.csv"
    pd.DataFrame({"Frame": [1, 2, 3], "Toy Code": [1, 1, 2]}).to_csv(
        frame_a, index=False
    )
    pd.DataFrame({"frame_number": [1, 2, 3], "object_code": [1, 2, 2]}).to_csv(
        frame_b, index=False
    )
    frame_result = run_pairwise_reliability(
        frame_a,
        frame_b,
        config=ReliabilityConfig(requested_domains=("toy",)),
        write_outputs=False,
    )
    assert frame_result.metadata["alignment"]["method"] == "exact_frame"
    assert frame_result.summary.loc[0, "disagreement_count"] == 1
    assert frame_result.disagreements["video_time"].isna().all()

    malformed_a = temp / "malformed_a.csv"
    malformed_b = temp / "malformed_b.csv"
    pd.DataFrame({"toy_code": [1, 2]}).to_csv(malformed_a, index=False)
    pd.DataFrame({"toy_code": [1, 2]}).to_csv(malformed_b, index=False)
    try:
        run_pairwise_reliability(
            malformed_a,
            malformed_b,
            config=ReliabilityConfig(requested_domains=("toy",)),
            write_outputs=False,
        )
    except ValueError as exc:
        assert "No safe shared alignment field" in str(exc)
    else:
        raise AssertionError("Malformed files should not produce reliability metrics.")

print("All synthetic reliability checks passed.")

Loading file A: 9001_9m_CoderA.csv
Loading file B: 9001_9m_CoderB.csv
Aligned 5 rows using nearest_video_time (tolerance=20.0).
toy: 75.0000% agreement, 1 disagreements, kappa=0.636363636.
looking: 75.0000% agreement, 1 disagreements, kappa=0.5.
Wrote 7 report file(s) to /private/var/folders/z4/kn9vw_0j55s4k90z98drdjpc0000gn/T/tmpz5w1bi5l/reports
Loading file A: frame_coder_a.csv
Loading file B: frame_coder_b.csv
Aligned 3 rows using exact_frame (tolerance=0.0).
toy: 66.6667% agreement, 1 disagreements, kappa=0.4.
Report writing is disabled for this run.
Loading file A: malformed_a.csv
Loading file B: malformed_b.csv


All synthetic reliability checks passed.


## Practical testing plan

### Before first operational use

1. **Toy fixtures:** hand-score a small two-coder file containing exact matches, disagreements, missing values, every valid toy label, numeric labels, and NO_TOY. Reconcile every summary count by hand.
2. **Looking fixtures:** separately test 0/1 attention codes, text aliases, missing codes, 9, and 999. Verify that toy exclusions do not change looking denominators and vice versa.
3. **Schema variants:** permute columns; add unused/blank columns; vary capitalization; test Paradigm versus TOY and Looking versus LOOKING; verify explicit overrides; require errors for ambiguous or absent mappings.
4. **Alignment variants:** test exact time, small bounded offsets, offsets just outside tolerance, unequal start/end ranges, frame fallback, duplicated keys, non-overlapping ranges, and explicitly enabled row order.
5. **Time-linked logs:** for every synthetic disagreement, verify both source rows, both time values, delta, seconds, and HH:MM:SS against the input.
6. **Special conditions:** verify code 7 at 9 months becomes FISH, at known non-9-month ages becomes POP, and remains unresolved with a warning at unknown age. Confirm the approved handling of 9 and 999.
7. **Output contract:** parse every written CSV/JSON, check exact headers and numeric types, and confirm empty disagreement logs retain headers.

### Historical regression

Run all four supplied pairs (1038, 1046, 5093, 5108). Review alignment coverage and warnings first. Compare transition boundaries and a sample of disagreements against the source video. The 1038 pair requires codebook-approved mappings for FGN/FNG/BEAD; do not infer those labels from frequency alone. Save approved expected summaries as regression fixtures.

### Acceptance criteria

- Hand-computed toy and looking counts, percentages, exclusions, and kappa match the notebook.
- No clearly mismatched participant/age or non-overlapping pair can generate a result in strict mode.
- Every eligible disagreement is traceable to source rows and, when available, video time.
- Re-running top to bottom produces the same summaries from unchanged inputs.

## Refactoring roadmap

### Immediate

1. Confirm the authoritative toy/object and looking/attention codebooks, especially 9, 999, FGN/FNG/BEAD, and the 9-month FISH/other-age POP rule.
2. Run and adjudicate the supplied historical pairs; record approved explicit mappings rather than embedding guesses.
3. Freeze the output columns above and store small de-identified acceptance fixtures.
4. Require reviewers to sign off warnings and alignment coverage before accepting kappa.

### Medium priority

1. Move the reusable functions into a versioned Python module with pytest tests; keep this notebook as the guided runner.
2. Add a schema-profile command that proposes mappings without running reliability.
3. Add codebook files (JSON/YAML) keyed by study, age, condition, and coder export version.
4. Add an optional exclusions-detail CSV and batch runner for a manifest of approved pairs.
5. Add interval-to-sample conversion for event-level onset/offset exports after its sampling policy is scientifically approved.

### Later enhancements

1. Add a local user interface with file pickers and mapping previews.
2. Add bootstrap confidence intervals and prevalence/bias diagnostics alongside kappa.
3. Add weighted kappa only for genuinely ordinal domains.
4. Add cryptographic input hashes, software-version capture, and signed run manifests for stronger auditability.
5. Add secure adjudication status fields that link resolutions back to the disagreement log.

## Architecture and handoff

The notebook now follows a modular path:

1. **File loader:** load_coding_file parses CSVs, preserves source rows, and profiles malformed alignment values.
2. **Schema mapper:** resolve_schema converts aliases or explicit overrides to canonical fields.
3. **Validator/alignment layer:** validate_comparability and align_coding_files verify identity, shared domains, keys, overlap, tolerance, coverage, and provenance.
4. **Comparator:** compare_domain normalizes and applies domain-specific eligibility independently.
5. **Kappa calculator:** calculate_cohens_kappa calculates observed agreement, chance expectation, and kappa directly from eligible pairs.
6. **Disagreement logger:** compare_domain constructs a review-ready log with both raw and normalized values and time/frame/source-row links.
7. **Report writer:** write_reliability_reports creates combined, domain-specific, and metadata artifacts.
8. **Orchestrator:** run_pairwise_reliability connects the modules and provides concise user-facing messages.

## Takeaways

- The workflow is explicit about what was compared, what was excluded, and how rows were aligned.
- Toy/object reliability and looking/attention reliability are never conflated.
- Time-linked logs support efficient video adjudication without discarding either coder's timestamp.
- Unsafe assumptions become errors or visible warnings; study-specific semantic mappings remain a research-team decision.
- The synthetic checks and executed historical example make the notebook rerunnable and reviewable in both local Jupyter and Colab environments.